In [1]:
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
import stlearn as st
st.settings.set_figure_params(dpi=800)
from PIL import Image, ImageChops, ImageDraw
import seaborn as sns
from skimage.color import rgb2hed
from scipy.ndimage.filters import gaussian_filter
import sys
from time import sleep
from tqdm.auto import tqdm
import anndata as ad
import seaborn as sns
from scipy import stats
from joblib import parallel_config, Parallel, delayed

/home/uqxtan9/micromamba/envs/spatialdata/lib/python3.10/site-packages/stlearn/tools/microenv/cci/het.py:192: NumbaDeprecationWarning: The keyword argument 'nopython=False' was supplied. From Numba 0.59.0 the default is being changed to True and use of 'nopython=False' will raise a warning as the argument will have no effect. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit(parallel=True, nopython=False)
/tmp/ipykernel_319771/1079766432.py:12: DeprecationWarning: Please import `gaussian_filter` from the `scipy.ndimage` namespace; the `scipy.ndimage.filters` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.ndimage.filters import gaussian_filter


In [2]:
import subprocess
system = subprocess.check_output(["hostname", "-s"]).decode("utf-8").strip()
BASE_PATH_ = Path()
if "bun" in system:
    BASE_PATH_ = Path("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/")
elif "imb-quan-gpu" in system:
    BASE_PATH_ = Path("/home/uqxtan9/Q1851/Xiao/")


BASE_PATH = BASE_PATH_ / "Working_project/MB"
DATA_PATH = BASE_PATH / "Xenium_Brain"
XENIUM_RAW_PATH = DATA_PATH / "Xenium_RAW"
MALDI_RAW_PATH = DATA_PATH / "MALDI_RAW/imzml_file"
PROCESSED = BASE_PATH / "PROCESSED"
PROCESSED.mkdir(exist_ok=True, parents=True)
OUT_PATH = BASE_PATH / "PLOTS" / "Xenium"
OUT_PATH.mkdir(exist_ok=True, parents=True)
QC_PATH = OUT_PATH / "QC"
QC_PATH.mkdir(exist_ok=True, parents=True)
CLS_PATH = OUT_PATH / "CLUSTERING"
CLS_PATH.mkdir(exist_ok=True, parents=True)
CCI_PATH = OUT_PATH / "CCI"
CCI_PATH.mkdir(exist_ok=True, parents=True)

In [21]:
OUT_PATH = Path("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/for_Tuan/Xenium")
QC_PATH = OUT_PATH / "QC"
QC_PATH.mkdir(exist_ok=True, parents=True)
CLS_PATH = OUT_PATH / "CLUSTERING"
CLS_PATH.mkdir(exist_ok=True, parents=True)
GENE_PATH = OUT_PATH / "GENE"
GENE_PATH.mkdir(exist_ok=True, parents=True)
MARKER_ANNO = OUT_PATH / "MARKER_ANNO"
MARKER_ANNO.mkdir(exist_ok=True, parents=True)

In [18]:
samples = {}
for file in XENIUM_RAW_PATH.glob("./*"):
    if file.is_dir():
        library_id = file.stem
        print(library_id)
        adata = st.ReadXenium(feature_cell_matrix_file=file / "cell_feature_matrix.h5",
                                cell_summary_file=file / "cells.csv.gz",
                                library_id=library_id,
                                scale=1,
                                spot_diameter_fullres=15 # Recommend
                                )
        df = pd.read_parquet(file / "cells.parquet")
        df.set_index(adata.obs_names, inplace=True)
        adata.obs = pd.concat([adata.obs, df], axis=1)
        adata.obs_names = adata.obs_names.map(lambda x: f"{x}-{library_id}")
        sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
        fig, axs = plt.subplots(1, 4, figsize=(15, 4))

        axs[0].set_title("Total transcripts per cell")
        sns.histplot(
            adata.obs["total_counts"],
            kde=False,
            ax=axs[0],
        )

        axs[1].set_title("Unique transcripts per cell")
        sns.histplot(
            adata.obs["n_genes_by_counts"],
            kde=False,
            ax=axs[1],
        )


        axs[2].set_title("Area of segmented cells")
        sns.histplot(
            adata.obs["cell_area"],
            kde=False,
            ax=axs[2],
        )

        axs[3].set_title("Nucleus ratio")
        sns.histplot(
            adata.obs["nucleus_area"] / adata.obs["cell_area"],
            kde=False,
            ax=axs[3],
        )
        # fig.savefig(QC_PATH / f"{library_id}_QC_hist.png", dpi=300, bbox_inches="tight")
        fig.savefig(QC_PATH / f"{library_id}_QC_hist.pdf", dpi=300, bbox_inches="tight")
        plt.close()
        samples[library_id] = adata

P6055125
P52407
Ctrl_1
Treated_1
P38685
Treated_2
Ctrl_2
B59460
P62560


In [19]:
sample_names = [f.stem for f in XENIUM_RAW_PATH.iterdir() if f.is_dir()]
sample_names

['P6055125',
 'P52407',
 'Ctrl_1',
 'Treated_1',
 'P38685',
 'Treated_2',
 'Ctrl_2',
 'B59460',
 'P62560']

In [20]:
samples = {}
for i, name in enumerate(sample_names):
    print(f"Processing {name} ({i+1}/{len(sample_names)})")
    adata= sc.read(PROCESSED / f"{name}_processed_marker_label.h5ad")
    samples[name] = adata

Processing P6055125 (1/9)
Processing P52407 (2/9)
Processing Ctrl_1 (3/9)
Processing Treated_1 (4/9)
Processing P38685 (5/9)
Processing Treated_2 (6/9)
Processing Ctrl_2 (7/9)
Processing B59460 (8/9)
Processing P62560 (9/9)


In [22]:
adata

AnnData object with n_obs × n_vars = 446985 × 266
    obs: 'imagecol', 'imagerow', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'cell_type'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'mt'
    uns: 'leiden', 'log1p', 'neighbors', 'pca', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [23]:
gene_dic = {
    "Tumour": ["EGFR", "IDH1", "IDH2", "PDGFRA", "TP53"],
    "Proliferation": ["PCNA", "TOP2A", "MKI67"],
    "Microglia" : ["CD68", "AIF1"],
    "T cell": ["CD4", "CD3G", "TRAC"],
    "Vascular": ["PECAM1", "CSPG4"],
    "Oligodendrocytes": ["OLIG1", "OLIG2"],
    "Astrocytes": ["AQP4"],
}

In [25]:
for name, adata in samples.items():
    for key, gene_ls in gene_dic.items():
        fig, ax = plt.subplots(figsize=(10, 5))
        st.pl.gene_plot(adata, gene_symbols=gene_ls, ax=ax, size=1, show_plot=False)
        ax.set_title(key)
        plt.tight_layout()
        fig.savefig(GENE_PATH / f"{name}_{key}_all_plot.png", dpi=300, bbox_inches="tight")
        plt.close()
        for gene in gene_ls:
            fig, ax = plt.subplots(figsize=(10, 5))
            st.pl.gene_plot(adata, gene_symbols=gene_ls, ax=ax, size=1, show_plot=False)
            ax.set_title(key)
            plt.tight_layout()
            fig.savefig(GENE_PATH / f"{name}_{key}_{gene}_plot.png", dpi=300, bbox_inches="tight")
            plt.close()

In [30]:
for name, adata in samples.items():
    fig, ax = plt.subplots(figsize=(10, 5))
    st.pl.cluster_plot(adata, use_label = "cell_type", size=1, ax=ax)
    ax.set_title("marker annotation")
    plt.tight_layout()
    fig.savefig(MARKER_ANNO / f"{name}_cell_type.png", dpi=300, bbox_inches="tight")
    plt.close()

/home/uqxtan9/micromamba/envs/spatialdata/lib/python3.10/site-packages/stlearn/plotting/classes.py:733: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for i, cluster in enumerate(self.query_adata.obs.groupby(self.use_label)):


In [32]:
for name, adata in samples.items():
    fig, ax = plt.subplots(figsize=(10, 5))
    st.pl.cluster_plot(adata, use_label = "leiden", size=1, ax=ax)
    ax.set_title("clustering")
    plt.tight_layout()
    fig.savefig(CLS_PATH / f"{name}_cell_type.png", dpi=300, bbox_inches="tight")
    plt.close()

/home/uqxtan9/micromamba/envs/spatialdata/lib/python3.10/site-packages/stlearn/plotting/classes.py:733: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for i, cluster in enumerate(self.query_adata.obs.groupby(self.use_label)):
